In [ ]:
# =============================================================================
# ADVANCED SOCIAL MEDIA METADATA ENGINE
# OPENROUTER FREE MODELS | GROQ FALLBACK | 4X CANDIDATES | RANKING
# GOOGLE DRIVE + GOOGLE SHEETS | SINGLE GOOGLE COLAB CELL
# =============================================================================

# GitHub Actions-compatible runtime (no Google Colab runtime required)
import os
import re
import sys
import json
import time
import csv
import subprocess
import requests
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import gspread
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from groq import Groq

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

WORKSPACE_ROOT = Path.cwd()
SOURCE_FOLDER_PATH = str((WORKSPACE_ROOT / os.getenv("PART3_OUTPUT_DIR", "Ready_!")).resolve())
THUMBNAIL_FOLDER_NAME = "thum"
REPORTS_FOLDER_ID = os.getenv("GOOGLE_DRIVE_REPORTS_FOLDER_ID", "").strip()
THUMBNAIL_FOLDER_ID = os.getenv("GOOGLE_DRIVE_THUMBNAIL_FOLDER_ID", "").strip()
SCHEDULE_TIMEZONE = ZoneInfo(os.getenv("SCHEDULE_TIMEZONE", "Asia/Karachi"))
LOCAL_OUTPUT_DIR = WORKSPACE_ROOT / "Part_4_Output"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POSTING_SCHEDULE = [
    "10:00 AM",
    "02:00 PM",
    "06:00 PM",
    "10:00 PM"
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Active and valid free models on OpenRouter
OPENROUTER_MODELS = [
    "google/gemma-2-9b-it:free",
    "meta-llama/llama-3-8b-instruct:free",
    "mistralai/mistral-7b-instruct:free",
    "qwen/qwen-2.5-72b-instruct:free",
    "meta-llama/llama-3.2-3b-instruct:free",
    "openrouter/free"
]

# Active free or standard models on Groq
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama3-8b-8192",
    "gemma2-9b-it",
    "mixtral-8x7b-32768"
]

SHEET_HEADER = [
    "Video Name",
    "Video Path",
    "Status",
    "Time_zone",
    "Video Link",
    "Thumbnail Link",
    "YT Title",
    "YT Description",
    "YT Tags",
    "TikTok Caption",
    "Instagram Caption",
    "FB Caption",
    "Popular Captions",
    "Hashtags (#)",
    "Trending Event",
    "Keywords",
    "Snap Chat",
    "AI Engine",
    "Ranking Used",
    "Winning Score",
    "Candidate Scores",
    "Generated At"
]


# =============================================================================
# 2. API KEYS
# =============================================================================

# API keys are supplied only through GitHub Actions Secrets.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

if not OPENROUTER_API_KEY and not GROQ_API_KEY:
    raise RuntimeError(
        "Missing API keys. Add OPENROUTER_API_KEY and/or GROQ_API_KEY "
        "to GitHub Settings > Secrets and variables > Actions."
    )

try:
    groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
except Exception as e:
    print(f"Warning initializing Groq client: {e}")
    groq_client = None


# =============================================================================
# 3. GOOGLE AUTHENTICATION (SERVICE ACCOUNT FOR GITHUB ACTIONS)
# =============================================================================

print("Authenticating Google service account...")
service_account_json = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON", "").strip()
if not service_account_json:
    raise RuntimeError("Missing GOOGLE_SERVICE_ACCOUNT_JSON GitHub secret.")
if not REPORTS_FOLDER_ID:
    raise RuntimeError("Missing GOOGLE_DRIVE_REPORTS_FOLDER_ID GitHub secret.")

try:
    service_account_info = json.loads(service_account_json)
except json.JSONDecodeError as exc:
    raise RuntimeError("GOOGLE_SERVICE_ACCOUNT_JSON is not valid JSON.") from exc

GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/spreadsheets",
]
creds = Credentials.from_service_account_info(
    service_account_info,
    scopes=GOOGLE_SCOPES,
)
gc = gspread.authorize(creds)
drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)

if not os.path.isdir(SOURCE_FOLDER_PATH):
    raise FileNotFoundError(
        f"Part 3 artifact folder not found: {SOURCE_FOLDER_PATH}. "
        "Confirm that the Part 3 artifact was downloaded into Ready_!."
    )

print("Google service account authentication successful.")
print("Part 3 source folder:", SOURCE_FOLDER_PATH)


# =============================================================================
# 4. GOOGLE DRIVE FOLDER HELPERS
# =============================================================================

def escape_drive_query(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def get_or_create_folder(name, parent_id="root"):
    safe_name = escape_drive_query(name)
    query = (
        f"name = '{safe_name}' and "
        f"'{parent_id}' in parents and "
        "mimeType = 'application/vnd.google-apps.folder' and "
        "trashed = false"
    )
    result = drive_service.files().list(
        q=query,
        fields="files(id,name,parents)",
        pageSize=100,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    folders = result.get("files", [])
    if folders:
        return folders[0]["id"]

    metadata = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
    }
    if parent_id != "root":
        metadata["parents"] = [parent_id]
    created = drive_service.files().create(
        body=metadata,
        fields="id",
        supportsAllDrives=True,
    ).execute()
    return created["id"]


def move_file_to_folder(file_id, folder_id):
    metadata = drive_service.files().get(
        fileId=file_id,
        fields="parents",
        supportsAllDrives=True,
    ).execute()
    parents = metadata.get("parents", [])
    if folder_id in parents:
        return
    previous = ",".join(parents)
    update_args = {
        "fileId": file_id,
        "addParents": folder_id,
        "fields": "id,parents",
        "supportsAllDrives": True,
    }
    if previous:
        update_args["removeParents"] = previous
    drive_service.files().update(**update_args).execute()


# Store thumbnails in the supplied Drive folder, or create a "thum" folder
# inside the reports folder when no separate thumbnail folder ID is configured.
if not THUMBNAIL_FOLDER_ID:
    THUMBNAIL_FOLDER_ID = get_or_create_folder(
        THUMBNAIL_FOLDER_NAME,
        REPORTS_FOLDER_ID,
    )
else:
    drive_service.files().get(
        fileId=THUMBNAIL_FOLDER_ID,
        fields="id,name,mimeType",
        supportsAllDrives=True,
    ).execute()


# =============================================================================
# 5. GOOGLE SHEET HELPERS
# =============================================================================

def get_or_create_report_sheet(folder_name):
    sheet_name = f"{folder_name}_Viral_Report"

    try:
        spreadsheet = gc.open(sheet_name)
        worksheet = spreadsheet.get_worksheet(0)

        print("Opening existing sheet:", sheet_name)

    except gspread.SpreadsheetNotFound:
        # Create directly inside the shared reports folder. This avoids relying
        # on storage in the service account's personal Drive root.
        created = drive_service.files().create(
            body={
                "name": sheet_name,
                "mimeType": "application/vnd.google-apps.spreadsheet",
                "parents": [REPORTS_FOLDER_ID],
            },
            fields="id",
            supportsAllDrives=True,
        ).execute()
        spreadsheet = gc.open_by_key(created["id"])
        worksheet = spreadsheet.get_worksheet(0)
        print("Created sheet:", sheet_name)

    # Keep existing report sheets inside the configured reports folder.
    move_file_to_folder(spreadsheet.id, REPORTS_FOLDER_ID)

    if worksheet.row_values(1) != SHEET_HEADER:
        worksheet.update(
            range_name="A1",
            values=[SHEET_HEADER],
            value_input_option="RAW"
        )

    return spreadsheet, worksheet


def get_last_post_time(worksheet):
    try:
        values = worksheet.col_values(4)

        for value in reversed(values[1:]):
            try:
                return datetime.strptime(
                    value, "%Y-%m-%d %I:%M %p"
                ).replace(tzinfo=SCHEDULE_TIMEZONE)
            except (ValueError, TypeError):
                continue

    except Exception as exc:
        print("Schedule read warning:", exc)

    return None


def get_next_schedule_slot(last_dt):
    schedule_times = sorted(
        datetime.strptime(t, "%I:%M %p").time()
        for t in POSTING_SCHEDULE
    )

    now = datetime.now(SCHEDULE_TIMEZONE)

    if last_dt is None:
        last_dt = now - timedelta(minutes=1)

    base_date = max(last_dt.date(), now.date())

    for schedule_time in schedule_times:
        candidate = datetime.combine(base_date, schedule_time, tzinfo=SCHEDULE_TIMEZONE)

        if candidate > last_dt and candidate > now:
            return candidate

    next_date = base_date + timedelta(days=1)

    return datetime.combine(
        next_date,
        schedule_times[0],
        tzinfo=SCHEDULE_TIMEZONE,
    )


# =============================================================================
# 6. JSON AND TEXT HELPERS
# =============================================================================

def extract_json(text):
    text = str(text).strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start < 0 or end <= start:
        raise ValueError("AI did not return a JSON object")

    return json.loads(text[start:end + 1])


def normalize_metadata(data):
    required_fields = [
        "yt_title",
        "yt_description",
        "yt_tags",
        "tiktok_caption",
        "instagram_caption",
        "fb_caption",
        "popular_captions",
        "hashtags",
        "trending_event",
        "keywords",
        "snapchat_caption"
    ]

    for field in required_fields:
        data.setdefault(field, "")

    for field in required_fields:
        value = data[field]

        if isinstance(value, list):
            value = ", ".join(map(str, value))

        data[field] = str(value).strip()[:49000]

    return data


# =============================================================================
# 7. OPENROUTER GENERATION
# =============================================================================

def openrouter_generate(prompt):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OpenRouter API key is missing")

    last_exception = None
    for model_candidate in OPENROUTER_MODELS:
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                    "HTTP-Referer": "https://github.com/",
                    "X-Title": "Social Media Metadata Ranking Engine"
                },
                json={
                    "model": model_candidate,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are an expert social-media metadata "
                                "writer. Return valid JSON only. "
                                "Do not fabricate facts or current trends."
                            )
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    "temperature": 0.7,
                    "max_tokens": 1800,
                    "stream": False
                },
                timeout=60
            )

            if response.ok:
                payload = response.json()
                content = payload["choices"][0]["message"]["content"]
                actual_model = payload.get("model", model_candidate)
                return normalize_metadata(extract_json(content)), actual_model
            else:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")
        except Exception as exc:
            last_exception = exc
            print(f"    OpenRouter model {model_candidate} failed: {exc}. Trying next fallback...")
            time.sleep(1.5)

    raise last_exception or RuntimeError("All configured OpenRouter models failed.")


# =============================================================================
# 8. GROQ GENERATION
# =============================================================================

def groq_generate(prompt):
    if not groq_client:
        raise RuntimeError("Groq API key or client is missing")

    last_exception = None
    for model_candidate in GROQ_MODELS:
        try:
            response = groq_client.chat.completions.create(
                model=model_candidate,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert social-media metadata "
                            "writer. Return valid JSON only. "
                            "Do not fabricate facts or current trends."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.7,
                max_tokens=1800,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            return normalize_metadata(extract_json(content)), model_candidate
        except Exception as exc:
            last_exception = exc
            print(f"    Groq model {model_candidate} failed: {exc}. Trying next fallback...")
            time.sleep(1.5)

    raise last_exception or RuntimeError("All configured Groq models failed.")


# =============================================================================
# 9. QUALITY SCORING ENGINE
# =============================================================================

def score_candidate(metadata, video_context):
    """
    Local quality heuristic.
    """
    score = 0.0

    title = metadata.get("yt_title", "")
    description = metadata.get("yt_description", "")
    tags = metadata.get("yt_tags", "")
    hashtags = metadata.get("hashtags", "")
    keywords = metadata.get("keywords", "")

    # Title readability and useful length
    if 30 <= len(title) <= 75:
        score += 20
    elif 15 <= len(title) <= 90:
        score += 12
    elif title:
        score += 5

    # Description usefulness
    if len(description) >= 100:
        score += 15
    elif len(description) >= 40:
        score += 10

    # Tags and hashtags
    if tags:
        score += 10

    hashtag_count = len(re.findall(r"#\w+", hashtags))

    if 3 <= hashtag_count <= 8:
        score += 15
    elif hashtag_count > 0:
        score += 8

    keyword_count = len([
        item for item in re.split(r"[,;\n]", keywords)
        if item.strip()
    ])

    if 4 <= keyword_count <= 12:
        score += 15
    elif keyword_count > 0:
        score += 8

    # Topic relevance using word overlap
    context_words = set(re.findall(r"\b[a-zA-Z0-9]{4,}\b", video_context.lower()))
    metadata_text = " ".join([title, description, tags, hashtags, keywords]).lower()
    metadata_words = set(re.findall(r"\b[a-zA-Z0-9]{4,}\b", metadata_text))

    if context_words:
        overlap = len(context_words & metadata_words) / len(context_words)
        score += min(20, overlap * 40)

    if not title:
        score -= 40

    words = re.findall(r"\b[a-zA-Z]{4,}\b", (title + " " + description).lower())
    if words:
        repetition = 1 - len(set(words)) / len(words)
        score -= min(15, repetition * 30)

    return round(max(0, min(100, score)), 2)


# =============================================================================
# 10. STRATEGIES AND 4X ENGINE WITH HYBRID FALLBACKS
# =============================================================================

STRATEGIES = [
    (
        "SEARCH_SEO",
        "Focus on accurate search phrases, relevant keywords, clear wording, and discoverability."
    ),
    (
        "CURIOSITY_HOOK",
        "Create a compelling truthful hook that encourages interest without misleading clickbait."
    ),
    (
        "RETENTION",
        "Focus on the audience's reason to watch, emotional connection, clarity, and strong opening wording."
    ),
    (
        "PLATFORM_OPTIMIZATION",
        "Focus on natural language, platform suitability, readability, and relevant hashtags."
    )
]


def build_prompt(video_name, platform, strategy_name, instructions):
    return f"""
Generate metadata for this video.

VIDEO FILE:
{video_name}

TARGET PLATFORM:
{platform}

AUDIENCE:
India and United Kingdom, where relevant to the actual content.
Use the language appropriate for the video.

STRATEGY:
{strategy_name}

INSTRUCTIONS:
{instructions}

RULES:
- Do not invent events, news, trends, facts, or people.
- Do not claim a hashtag is trending without verified trend data.
- Use accurate and relevant keywords.
- Avoid keyword stuffing and repetitive hashtags.
- Write naturally and professionally.
- Use suitable emojis where appropriate.
- Make platform-specific captions distinct.
- If the content is unknown from the filename, do not pretend that you watched the video.
- Trending Event should say "No live trend data verified" unless actual verified trend information was supplied.

Return ONLY valid JSON with these exact keys:
{{
  "yt_title": "YouTube title",
  "yt_description": "YouTube description",
  "yt_tags": "comma-separated tags",
  "tiktok_caption": "TikTok caption",
  "instagram_caption": "Instagram caption",
  "fb_caption": "Facebook caption",
  "popular_captions": "alternative caption ideas",
  "hashtags": "#relevant #hashtags",
  "trending_event": "verified event or no live trend data verified",
  "keywords": "comma-separated keywords",
  "snapchat_caption": "Snapchat caption"
}}
"""


def generate_4x_ranked_metadata(video_name):
    context = f"Video filename: {video_name}"

    # 1. TRY OPENROUTER 4X PLAN FIRST
    print("\n  [Plan A] Running OpenRouter 4x candidates generation and ranking...")
    candidates = []
    openrouter_failed = False
    for number, (strategy, instructions) in enumerate(STRATEGIES, start=1):
        print(f"    OpenRouter candidate {number}/4: {strategy}")
        prompt = build_prompt(video_name, "Multi-platform social media", strategy, instructions)

        candidate = None
        for attempt in range(2):
            try:
                candidate, model_used = openrouter_generate(prompt)
                candidate["strategy"] = strategy
                candidate["model_used"] = model_used
                candidate["score"] = score_candidate(candidate, context)
                break
            except Exception as exc:
                print(f"      Attempt {attempt+1} failed: {exc}")
                time.sleep(1.5)

        if candidate is None:
            print("    OpenRouter 4x generation incomplete/failed. Switching to Groq 4x Fallback.")
            openrouter_failed = True
            break
        candidates.append(candidate)
        time.sleep(0.5)

    if not openrouter_failed and len(candidates) == 4:
        # Successfully ranked with OpenRouter
        candidates.sort(key=lambda item: item["score"], reverse=True)
        winner = candidates[0]
        scores = " | ".join(f'{item["strategy"]}:{item["score"]}' for item in candidates)
        print(f"    OpenRouter 4x Winner: {winner['strategy']} (Score: {winner['score']})")
        return {
            "status": "openrouter_4x_ranked",
            "ranking_used": True,
            "selected": winner,
            "score": winner["score"],
            "candidate_scores": scores,
            "model_used": winner.get("model_used", ""),
            "error": ""
        }

    # 2. TRY GROQ 4X PLAN (IF OPENROUTER FAILS)
    print("\n  [Plan B] Running Groq Free API 4x candidates generation and ranking fallback...")
    candidates = []
    groq_failed = False
    for number, (strategy, instructions) in enumerate(STRATEGIES, start=1):
        print(f"    Groq candidate {number}/4: {strategy}")
        prompt = build_prompt(video_name, "Multi-platform social media", strategy, instructions)

        candidate = None
        for attempt in range(2):
            try:
                candidate, model_used = groq_generate(prompt)
                candidate["strategy"] = strategy
                candidate["model_used"] = model_used
                candidate["score"] = score_candidate(candidate, context)
                break
            except Exception as exc:
                print(f"      Groq Attempt {attempt+1} failed: {exc}")
                time.sleep(1.5)

        if candidate is None:
            print("    Groq 4x generation incomplete/failed.")
            groq_failed = True
            break
        candidates.append(candidate)
        time.sleep(0.5)

    if not groq_failed and len(candidates) == 4:
        # Successfully ranked with Groq
        candidates.sort(key=lambda item: item["score"], reverse=True)
        winner = candidates[0]
        scores = " | ".join(f'{item["strategy"]}:{item["score"]}' for item in candidates)
        print(f"    Groq 4x Winner: {winner['strategy']} (Score: {winner['score']})")
        return {
            "status": "groq_4x_ranked",
            "ranking_used": True,
            "selected": winner,
            "score": winner["score"],
            "candidate_scores": scores,
            "model_used": winner.get("model_used", ""),
            "error": ""
        }

    # 3. BOTH 4X PLANS FAILED: GENERATE SHEET WITHOUT 4X RANKING
    print("\n  [Plan C] Both OpenRouter & Groq 4x plans failed. Running single last-resort single generation fallback...")
    fallback_prompt = build_prompt(
        video_name,
        "Multi-platform social media",
        "BALANCED_FALLBACK",
        "Generate one single balanced, highly accurate metadata set immediately without multiple candidates."
    )

    # Try Groq for single fallback
    try:
        print("    Attempting single fallback generation via Groq...")
        fallback, model_used = groq_generate(fallback_prompt)
        return {
            "status": "groq_single_fallback",
            "ranking_used": False,
            "selected": fallback,
            "score": "",
            "candidate_scores": "",
            "model_used": model_used,
            "error": ""
        }
    except Exception as groq_exc:
        print(f"    Groq single fallback failed: {groq_exc}")

    # Try OpenRouter for single fallback
    try:
        print("    Attempting single fallback generation via OpenRouter...")
        fallback, model_used = openrouter_generate(fallback_prompt)
        return {
            "status": "openrouter_single_fallback",
            "ranking_used": False,
            "selected": fallback,
            "score": "",
            "candidate_scores": "",
            "model_used": model_used,
            "error": ""
        }
    except Exception as or_exc:
        print(f"    OpenRouter single fallback failed: {or_exc}")

    return {
        "status": "failed",
        "ranking_used": False,
        "selected": {},
        "score": "",
        "candidate_scores": "",
        "model_used": "",
        "error": "All OpenRouter and Groq fallbacks failed completely."
    }


# =============================================================================
# 11. THUMBNAIL CREATION
# =============================================================================

def create_thumbnail(video_path, video_name):
    temporary_path = str(
        LOCAL_OUTPUT_DIR / f"thumb_{int(time.time() * 1000)}.jpg"
    )
    try:
        duration_text = subprocess.check_output(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                video_path,
            ],
            text=True,
        ).strip()
        duration = float(duration_text)
        timestamp = min(1.0, max(0.0, duration / 2.0))
        subprocess.run(
            [
                "ffmpeg", "-y", "-ss", str(timestamp), "-i", video_path,
                "-frames:v", "1", "-q:v", "2", temporary_path,
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        if not os.path.isfile(temporary_path) or os.path.getsize(temporary_path) == 0:
            raise RuntimeError("FFmpeg did not create a thumbnail image.")

        metadata = {
            "name": f"{Path(video_name).stem}_thumb.jpg",
            "parents": [THUMBNAIL_FOLDER_ID],
        }
        media = MediaFileUpload(temporary_path, mimetype="image/jpeg", resumable=True)
        uploaded = drive_service.files().create(
            body=metadata,
            media_body=media,
            fields="id,webViewLink",
            supportsAllDrives=True,
        ).execute()
        return uploaded.get(
            "webViewLink",
            f"https://drive.google.com/file/d/{uploaded['id']}/view",
        )
    except Exception as exc:
        print("  Thumbnail warning:", exc)
        return "N/A"
    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)


# =============================================================================
# 12. MAIN PROCESS
# =============================================================================

print("\nStarting Advanced Hybrid Metadata Engine...")
print("Source:", SOURCE_FOLDER_PATH)
print("Google Sheets reports folder ID:", REPORTS_FOLDER_ID)

if not any(
    name.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
    for _, _, filenames in os.walk(SOURCE_FOLDER_PATH)
    for name in filenames
):
    raise RuntimeError(
        f"No supported videos were found in {SOURCE_FOLDER_PATH}. "
        "Check that Part 3 uploaded its Ready_! output artifact."
    )

processed_count = 0
skipped_count = 0
failed_count = 0
local_rows = []
report_links = []

for root, dirs, files in os.walk(SOURCE_FOLDER_PATH):

    # Avoid scanning thumbnail folder
    dirs[:] = [
        folder for folder in dirs
        if folder != THUMBNAIL_FOLDER_NAME
    ]

    videos = sorted([
        name for name in files
        if name.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
    ])

    if not videos:
        continue

    folder_name = os.path.basename(root)

    spreadsheet, worksheet = get_or_create_report_sheet(
        folder_name
    )

    print("\nReport:", spreadsheet.title)
    print("Folder:", root)

    existing_names = set(
        worksheet.col_values(1)[1:]
    )

    last_dt = get_last_post_time(worksheet)
    rows_to_save = []

    for video_name in videos:

        if video_name in existing_names:
            print("  Skipping existing video:", video_name)
            skipped_count += 1
            continue

        video_path = os.path.join(root, video_name)

        next_dt = get_next_schedule_slot(last_dt)
        formatted_time = next_dt.strftime(
            "%Y-%m-%d %I:%M %p"
        )
        last_dt = next_dt

        print("\nProcessing:", video_name)
        print("Scheduled time:", formatted_time)

        # The source file comes from the Part 3 GitHub Actions artifact,
        # not a Google Drive file, so no Drive video URL is available here.
        video_url = "N/A - source is a GitHub Actions artifact"

        # Try multi-level candidates framework (OR 4X -> Groq 4X -> Single Fallback)
        try:
            result = generate_4x_ranked_metadata(video_name)

            if result["status"] == "failed":
                raise RuntimeError(result.get("error", "Generation failed completely."))

            metadata = result["selected"]

        except Exception as exc:
            print("  Metadata generation failed completely:", exc)
            failed_count += 1
            continue

        thumbnail_url = create_thumbnail(
            video_path,
            video_name
        )

        generated_at = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row = [
            video_name,
            video_path,
            "pending",
            formatted_time,
            video_url,
            thumbnail_url,
            metadata.get("yt_title", ""),
            metadata.get("yt_description", ""),
            metadata.get("yt_tags", ""),
            metadata.get("tiktok_caption", ""),
            metadata.get("instagram_caption", ""),
            metadata.get("fb_caption", ""),
            metadata.get("popular_captions", ""),
            metadata.get("hashtags", ""),
            metadata.get("trending_event", ""),
            metadata.get("keywords", ""),
            metadata.get("snapchat_caption", ""),
            result.get("model_used", ""),
            str(result.get("ranking_used", False)),
            str(result.get("score", "")),
            result.get("candidate_scores", ""),
            generated_at
        ]

        rows_to_save.append(row)
        local_rows.append(row)
        existing_names.add(video_name)
        processed_count += 1

        print("  AI status:", result["status"])
        print("  Ranking used:", result["ranking_used"])

    if rows_to_save:
        worksheet.append_rows(
            rows_to_save,
            value_input_option="USER_ENTERED"
        )

        print(
            f"Saved {len(rows_to_save)} rows to "
            f"{spreadsheet.title}"
        )

    sheet_url = f"https://docs.google.com/spreadsheets/d/{spreadsheet.id}/edit"
    report_links.append({"folder": folder_name, "spreadsheet": spreadsheet.title, "url": sheet_url})
    print("Google Sheet:", sheet_url)


# Save a local copy of newly generated rows and a run summary.
csv_path = LOCAL_OUTPUT_DIR / "metadata_rows.csv"
with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(SHEET_HEADER)
    writer.writerows(local_rows)

summary = {
    "processed_count": processed_count,
    "skipped_count": skipped_count,
    "failed_count": failed_count,
    "source_folder": SOURCE_FOLDER_PATH,
    "schedule_timezone": str(SCHEDULE_TIMEZONE),
    "google_sheets_reports": report_links,
    "generated_at": datetime.now(SCHEDULE_TIMEZONE).isoformat(),
}
summary_path = LOCAL_OUTPUT_DIR / "run_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + "=" * 65)
print("PROCESS COMPLETED")
print("=" * 65)
print("Successfully processed:", processed_count)
print("Skipped existing videos:", skipped_count)
print("Failed videos:", failed_count)
print("Local CSV report:", csv_path)
print("Local run summary:", summary_path)
print("Google Sheets reports:", len(report_links))

if processed_count == 0 and failed_count > 0:
    raise RuntimeError("All new video metadata generation attempts failed. Check API quotas and keys.")